<a href="https://colab.research.google.com/github/MohitJo27/Dissertation/blob/main/dissertation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assessing Security Risks in Data Lake Architectures
## Using Prowler & AWS Security Hub — Multi-Tool Cloud-Native Vulnerability Assessment

---

**Author:** Mohit | **Institution:** UPES | **Tool:** Prowler v3.11.3  
**Cloud:** AWS ap-south-1 (Mumbai) | **Account:** 673538734257 | **Date:** March 2026

---

## Notebook Structure

| # | Section | Purpose |
|---|---|---|
| 1 | Install & Import Libraries | Setup environment |
| 2 | Load & Combine CSVs | Read 3 Prowler scan exports |
| 3 | Pre-processing & Feature Engineering | Assign layers, compute Risk Index |
| 4 | Descriptive Statistics | Severity distributions, layer breakdowns |
| 5 | Chart 1 — Severity Distribution | Visualise finding counts by severity |
| 6 | Chart 2 — Layer Comparison | IAM vs S3 severity profiles |
| 7 | Chart 3 — Scan Progression | Baseline → Partial → Weakened |
| 8 | Chart 4 — Top 10 Risk Index | Ranked remediation priorities |
| 9 | Chart 5 — CVSS vs Exposure Scatter | Correlation visualisation |
| 10 | Statistical Tests | Chi-square, Pearson, Spearman, Regression |
| 11 | Risk Index Prioritisation | Top findings table |
| 12 | Business Impact Analysis | Breach cost & ROI |

---

### Risk Index Formula

$$\\text{Risk Index} = \\text{CVSS Score} \\times \\text{Exposure Level} \\times \\text{Architectural Weight}$$

| Parameter | Values |
|---|---|
| CVSS Score | Critical=9.5, High=7.5, Medium=5.5, Low=2.5 |
| Exposure Level | 1=Internal, 2=Cross-account, 3=Public |
| Architectural Weight | IAM=1.5, S3=1.2, Other=1.0 |


---
## Install Required Libraries

Run this once to install all dependencies. Comment it out after the first run.  
Libraries used:
- **pandas / numpy** — data manipulation and numerical operations
- **plotly** — interactive charts rendered in Jupyter
- **scipy** — chi-square test, Pearson and Spearman correlation coefficients
- **scikit-learn** — linear regression model and evaluation metrics



In [1]:
import subprocess, sys

packages = ['pandas', 'numpy', 'plotly', 'scipy', 'scikit-learn']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All libraries installed successfully!')

All libraries installed successfully!


---
## Import Libraries & Configure Settings

We configure Plotly to use the clean `plotly_white` template throughout the notebook for a consistent, publication-ready aesthetic.


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.stats import chi2_contingency, pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set default Plotly theme — clean white background for all charts
pio.templates.default = 'plotly_white'

# Colour palette — consistent across all charts
COLOR_SEV = {
    'critical': '#A32D2D',
    'high'    : '#BA7517',
    'medium'  : '#185FA5',
    'low'     : '#3B6D11'
}
COLOR_LAYER = {
    'IAM Access Control': '#A32D2D',
    'S3 Storage'        : '#185FA5',
    'Other'             : '#888780'
}
SEV_ORDER = ['critical', 'high', 'medium', 'low']

import plotly
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'plotly  : {plotly.__version__}')
print('Ready!')

pandas  : 2.2.2
numpy   : 2.0.2
plotly  : 5.24.1
Ready!


---
## Load and Combine the 3 Prowler Scan CSV Files

### Scan Configuration States

| Scan File | State | Misconfigurations Active |
|---|---|---|
| `scan1_baseline_csv.csv` | **Baseline** | None — all secure settings |
| `scan2_partial.csv` | **Partially Weakened** | Encryption disabled on 3 buckets, versioning suspended, IAM admin added |
| `scan3_weakened.csv` | **Fully Weakened** | All above + 3 more buckets public access removed |

### Why Semicolon Delimiter?
Prowler CSVs use `;` as the field separator (not `,`) because the description and remediation fields contain commas. The `latin-1` encoding handles special characters in the remediation text.

> **Update the file paths below** if your CSV files are in a different folder.


In [4]:
# ── File paths — update to match your saved location ────────────────────────
SCAN1_PATH = '/content/scan1_baseline.csv'   # Baseline scan
SCAN2_PATH = '/content/scan2_partial.csv'         # Partially weakened
SCAN3_PATH = '/content/scan3_weakened.csv'        # Fully weakened

# ── Load with semicolon separator and latin-1 encoding ───────────────────────
enc, sep = 'latin-1', ';'
df1 = pd.read_csv(SCAN1_PATH, sep=sep, encoding=enc, on_bad_lines='skip')
df2 = pd.read_csv(SCAN2_PATH, sep=sep, encoding=enc, on_bad_lines='skip')
df3 = pd.read_csv(SCAN3_PATH, sep=sep, encoding=enc, on_bad_lines='skip')

# ── Tag each row with its scan state ─────────────────────────────────────────
df1['scan_state'] = 'Baseline'
df2['scan_state'] = 'Partial'
df3['scan_state'] = 'Weakened'

# ── Combine into single master dataframe ─────────────────────────────────────
raw = pd.concat([df1, df2, df3], ignore_index=True)
raw['SEVERITY'] = raw['SEVERITY'].str.strip().str.lower()
raw['STATUS']   = raw['STATUS'].str.strip().str.upper()

print(f'Total records loaded : {len(raw)}')
print(f'Columns              : {len(raw.columns)}')
print(f'\nStatus counts:')
print(raw['STATUS'].value_counts().to_string())
print(f'\nRecords per scan state:')
print(raw.groupby('scan_state').size().to_string())
raw[['SEVERITY','STATUS','SERVICE_NAME','CHECK_TITLE','scan_state']].head(5)

Total records loaded : 969
Columns              : 39

Status counts:
STATUS
PASS    540
FAIL    429

Records per scan state:
scan_state
Baseline    323
Partial     323
Weakened    323


,SEVERITY,STATUS,SERVICE_NAME,CHECK_TITLE,scan_state
0,high,PASS,iam,Ensure users of groups with AdministratorAcces...,Baseline
1,high,PASS,iam,Ensure users of groups with AdministratorAcces...,Baseline
2,high,PASS,iam,Ensure IAM AWS-Managed policies that allow ful...,Baseline
3,high,PASS,iam,Ensure IAM AWS-Managed policies that allow ful...,Baseline
4,high,PASS,iam,Ensure IAM AWS-Managed policies that allow ful...,Baseline


---
## Pre-processing and Feature Engineering

This cell creates five new columns that are central to the analysis:

### 1. Architectural Layer
Each finding is labelled as `IAM Access Control` or `S3 Storage` based on the `SERVICE_NAME` field. This allows layer-stratified analysis.

### 2. CVSS Score
Prowler uses severity labels (critical/high/medium/low) rather than numeric CVSS scores. We map each label to its CVSS v3.0 midpoint value using a standard mapping.

### 3. Exposure Level (1–3)
A custom ordinal variable assessing how broadly accessible the vulnerable resource is:
- **1 = Internal** — accessible only within the account
- **2 = Cross-account** — accessible from other AWS accounts
- **3 = Public** — accessible from the public internet

### 4. Architectural Importance Weight
Reflects the relative criticality of each layer to data lake security:
- **IAM = 1.5** — identity compromise enables lateral movement across all layers
- **S3 = 1.2** — storage compromise directly exposes data assets
- **Other = 1.0** — baseline weight

### 5. Risk Index
The composite prioritisation score:
$$\\text{Risk Index} = \\text{CVSS} \\times \\text{Exposure} \\times \\text{Weight}$$
Maximum possible value = 9.5 × 3 × 1.5 = **42.75**


In [5]:
# ── Keep only FAIL records ───────────────────────────────────────────────────
fails = raw[raw['STATUS'] == 'FAIL'].copy()
print(f'Total FAIL records across all 3 scans: {len(fails)}')

# ── 1. Architectural Layer ────────────────────────────────────────────────────
def assign_layer(svc):
    s = str(svc).lower()
    if 'iam' in s: return 'IAM Access Control'
    if 's3'  in s: return 'S3 Storage'
    return 'Other'

fails['arch_layer'] = fails['SERVICE_NAME'].apply(assign_layer)

# ── 2. CVSS Score mapping ─────────────────────────────────────────────────────
sev_map = {'critical': 9.5, 'high': 7.5, 'medium': 5.5, 'low': 2.5}
fails['cvss_score'] = fails['SEVERITY'].map(sev_map)

# ── 3. Exposure Level ─────────────────────────────────────────────────────────
def assign_exposure(row):
    t = (str(row.get('STATUS_EXTENDED', '')) +
         str(row.get('CHECK_TITLE', ''))).lower()
    if 'public' in t: return 3
    if 'cross'  in t: return 2
    return 1

fails['exposure_level'] = fails.apply(assign_exposure, axis=1)

# ── 4. Architectural Weight ───────────────────────────────────────────────────
w_map = {'IAM Access Control': 1.5, 'S3 Storage': 1.2, 'Other': 1.0}
fails['arch_weight'] = fails['arch_layer'].map(w_map)

# ── 5. Risk Index ─────────────────────────────────────────────────────────────
fails['risk_index'] = fails['cvss_score'] * fails['exposure_level'] * fails['arch_weight']

# ── Baseline subset for single-scan analysis ─────────────────────────────────
base = fails[fails['scan_state'] == 'Baseline'].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\nBaseline FAIL records   : {len(base)}')
print(f'Architectural layers    :\n{base["arch_layer"].value_counts().to_string()}')
print(f'\nCVSS Score range        : {base["cvss_score"].min()} – {base["cvss_score"].max()}')
print(f'Exposure Level counts   :\n{base["exposure_level"].value_counts().sort_index().to_string()}')
print(f'\nRisk Index: min={base["risk_index"].min():.1f}, max={base["risk_index"].max():.1f}, mean={base["risk_index"].mean():.2f}')
print('\nFeature engineering complete!')

Total FAIL records across all 3 scans: 429

Baseline FAIL records   : 140
Architectural layers    :
arch_layer
S3 Storage            78
IAM Access Control    62

CVSS Score range        : 2.5 – 9.5
Exposure Level counts   :
exposure_level
1    121
2     16
3      3

Risk Index: min=3.0, max=34.2, mean=8.63

Feature engineering complete!


---
## Descriptive Statistics

Before visual analysis, we compute summary statistics that will populate the dissertation tables.

**Key metrics reported:**
- Frequency counts and percentages for each severity level
- Layer-stratified breakdowns (IAM vs S3)
- Scan progression showing total failures across all 3 states
- CVSS score and Risk Index central tendency measures

These numbers map directly to **Tables 4.1, 4.2, and 4.3** in the dissertation Results chapter.


In [6]:
print('=' * 65)
print('DESCRIPTIVE STATISTICS — DISSERTATION RESULTS CHAPTER')
print('=' * 65)

# ── Table 4.1: Severity Distribution ─────────────────────────────────────────
print('\nTABLE 4.1 — Severity Distribution (Baseline, n=140)')
print(f'{"Severity":<12} {"Count":>7} {"Percentage":>12} {"Cumulative":>12}')
print('-' * 46)
total   = len(base)
cumsum  = 0
for sev in SEV_ORDER:
    c = len(base[base['SEVERITY'] == sev])
    pct   = c / total * 100
    cumsum += pct
    print(f'{sev.capitalize():<12} {c:>7} {pct:>11.1f}% {cumsum:>11.1f}%')
print(f'{"TOTAL":<12} {total:>7} {"100.0%":>12}')

# ── Table 4.2: Layer breakdown ────────────────────────────────────────────────
print('\nTABLE 4.2 — Severity by Architectural Layer (Baseline)')
print(f'{"Severity":<12} {"IAM (n)":>10} {"IAM (%)":>10} {"S3 (n)":>10} {"S3 (%)":>10}')
print('-' * 54)
iam_sub = base[base['arch_layer'] == 'IAM Access Control']
s3_sub  = base[base['arch_layer'] == 'S3 Storage']
for sev in SEV_ORDER:
    c_iam = len(iam_sub[iam_sub['SEVERITY'] == sev])
    c_s3  = len(s3_sub[s3_sub['SEVERITY']  == sev])
    p_iam = c_iam / len(iam_sub) * 100 if len(iam_sub) > 0 else 0
    p_s3  = c_s3  / len(s3_sub)  * 100 if len(s3_sub)  > 0 else 0
    print(f'{sev.capitalize():<12} {c_iam:>10} {p_iam:>9.1f}% {c_s3:>10} {p_s3:>9.1f}%')

# ── Scan progression ──────────────────────────────────────────────────────────
print('\nSCAN PROGRESSION SUMMARY')
print(f'{"State":<12} {"Failures":>10} {"Passes":>10} {"Fail %":>10} {"Change":>10}')
print('-' * 54)
prev = None
for state in ['Baseline', 'Partial', 'Weakened']:
    f = len(fails[fails['scan_state'] == state])
    p = len(raw[raw['scan_state'] == state]) - f
    pct = f / (f + p) * 100
    ch = f'{"+"}{f-prev}' if prev is not None else '—'
    print(f'{state:<12} {f:>10} {p:>10} {pct:>9.1f}% {ch:>10}')
    prev = f

# ── CVSS and Risk Index ────────────────────────────────────────────────────────
print('\nCVSS SCORE SUMMARY (Baseline)')
print(base['cvss_score'].describe().round(2).to_string())
print('\nRISK INDEX SUMMARY (Baseline)')
print(base['risk_index'].describe().round(2).to_string())

DESCRIPTIVE STATISTICS — DISSERTATION RESULTS CHAPTER

TABLE 4.1 — Severity Distribution (Baseline, n=140)
Severity       Count   Percentage   Cumulative
----------------------------------------------
Critical           1         0.7%         0.7%
High              21        15.0%        15.7%
Medium            89        63.6%        79.3%
Low               29        20.7%       100.0%
TOTAL            140       100.0%

TABLE 4.2 — Severity by Architectural Layer (Baseline)
Severity        IAM (n)    IAM (%)     S3 (n)     S3 (%)
------------------------------------------------------
Critical              0       0.0%          1       1.3%
High                 20      32.3%          1       1.3%
Medium               26      41.9%         63      80.8%
Low                  16      25.8%         13      16.7%

SCAN PROGRESSION SUMMARY
State          Failures     Passes     Fail %     Change
------------------------------------------------------
Baseline            140        183      43.

---
## Chart 1: Severity Distribution

**Research Question addressed:** RQ1 — *What types and categories of security vulnerabilities exist within the data lake architecture?*

**What this chart shows:** The frequency distribution of all 140 vulnerability findings by severity level in the Scan 1 Baseline dataset.

**Key findings to highlight in dissertation:**
- Medium severity accounts for 63.6% of all findings — primarily S3 configuration hygiene issues
- High severity (15%) represents IAM governance failures requiring prioritised remediation
- The single Critical finding (public S3 bucket) represents the highest business impact item
- Mean CVSS = 5.21 places the average finding solidly in the medium-high risk band


In [7]:
# ── Chart 1: Severity Distribution Bar Chart ────────────────────────────────
counts_c1 = base['SEVERITY'].value_counts().reindex(SEV_ORDER, fill_value=0).reset_index()
counts_c1.columns = ['Severity', 'Count']
counts_c1['Percentage'] = (counts_c1['Count'] / counts_c1['Count'].sum() * 100).round(1)
counts_c1['Label']      = counts_c1['Count'].astype(str) + '<br>(' + counts_c1['Percentage'].astype(str) + '%)'
counts_c1['Color']      = counts_c1['Severity'].map(COLOR_SEV)

fig1 = go.Figure()
fig1.add_trace(go.Bar(
    x            = counts_c1['Severity'].str.capitalize(),
    y            = counts_c1['Count'],
    marker_color = counts_c1['Color'],
    marker_line  = dict(width=0),
    text         = counts_c1['Label'],
    textposition = 'outside',
    textfont     = dict(size=13, color='#333333'),
    width        = 0.5,
    hovertemplate = '<b>%{x}</b><br>Count: %{y}<br>Percentage: %{text}<extra></extra>'
))

fig1.update_layout(
    title    = dict(
        text = '<b>Vulnerability Severity Distribution — Scan 1 Baseline</b><br>'
               '<sup>n=140 failure findings | Cloud-based data lake architecture | AWS ap-south-1</sup>',
        font = dict(size=17), x=0.02
    ),
    xaxis    = dict(title='<b>Severity Level</b>', tickfont=dict(size=14)),
    yaxis    = dict(title='<b>Number of Findings</b>', tickfont=dict(size=13),
                   range=[0, counts_c1['Count'].max() * 1.28],
                   gridcolor='#F0F0F0'),
    plot_bgcolor = 'white',
    paper_bgcolor= 'white',
    height   = 500,
    showlegend = False,
    margin   = dict(t=100, b=70, l=70, r=40)
)

fig1.show()

---
## Cell 7 — Chart 2: Vulnerability Distribution by Architectural Layer

**Research Question addressed:** RQ3 — *Which architectural layer exhibits the highest concentration of critical and high-severity vulnerabilities?*

**What this chart shows:** A side-by-side grouped bar chart comparing severity profiles between the IAM Access Control layer and the S3 Storage layer.

**Key findings:**
- **IAM layer** dominates at High severity: 20 High findings vs S3's 1 High finding
- **S3 layer** is the sole source of the Critical finding (public bucket exposure)
- S3 contributes a substantially larger volume of Medium findings (63 vs 26)
- This asymmetry is statistically significant (chi-square test in Cell 10, p < 0.001)
- **Dissertation answer to RQ3:** The IAM Access Control layer exhibits the highest concentration of high-severity findings and is therefore the primary remediation priority


In [8]:
# ── Chart 2: Layer Comparison Grouped Bar ────────────────────────────────────
layer_rows = []
for layer in ['IAM Access Control', 'S3 Storage']:
    sub = base[base['arch_layer'] == layer]
    for sev in SEV_ORDER:
        c = len(sub[sub['SEVERITY'] == sev])
        layer_rows.append({
            'Layer': layer,
            'Severity': sev.capitalize(),
            'Count': c,
            'Pct': round(c / len(sub) * 100, 1) if len(sub) > 0 else 0
        })
layer_df = pd.DataFrame(layer_rows)

fig2 = go.Figure()
for layer in ['IAM Access Control', 'S3 Storage']:
    sub = layer_df[layer_df['Layer'] == layer]
    fig2.add_trace(go.Bar(
        name         = layer,
        x            = sub['Severity'],
        y            = sub['Count'],
        marker_color = COLOR_LAYER[layer],
        marker_line  = dict(width=0),
        text         = sub['Count'].astype(str) + '<br>(' + sub['Pct'].astype(str) + '%)',
        textposition = 'outside',
        textfont     = dict(size=11),
        width        = 0.35,
        hovertemplate= f'<b>{layer}</b><br>Severity: %{{x}}<br>Count: %{{y}}<extra></extra>'
    ))

fig2.update_layout(
    title    = dict(
        text = '<b>Vulnerability Severity Distribution by Architectural Layer</b><br>'
               '<sup>IAM Access Control vs S3 Storage — Scan 1 Baseline</sup>',
        font = dict(size=17), x=0.02
    ),
    xaxis    = dict(title='<b>Severity Level</b>', tickfont=dict(size=13)),
    yaxis    = dict(title='<b>Number of Findings</b>', tickfont=dict(size=13),
                   range=[0, 75], gridcolor='#F0F0F0'),
    barmode  = 'group',
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    height   = 500,
    legend   = dict(title='<b>Layer</b>', font=dict(size=12),
                    bgcolor='rgba(255,255,255,0.9)',
                    bordercolor='#CCCCCC', borderwidth=1),
    margin   = dict(t=100, b=70, l=70, r=40)
)

fig2.show()

---
## Chart 3: Scan Progression Across 3 Configuration States

**Research Question addressed:** RQ2 — *How effectively does Prowler detect and classify vulnerabilities across configuration states?*

**What this chart shows:** How vulnerability counts change as misconfigurations are progressively introduced from Baseline → Partial → Fully Weakened states.

**Key findings:**
- Total failures increase monotonically: 140 → 143 → 146 across the three states
- The increase is entirely driven by S3 Medium findings: 63 → 66 → 69 (+3 per state)
- IAM findings remain constant (62) — the IAM misconfigurations we introduced were already captured by existing checks
- This progression validates the experimental design and demonstrates Prowler's sensitivity to configuration changes
- The consistent 3-finding increase per S3 weakening step shows a predictable, linear detection pattern


In [9]:
# ── Chart 3: Scan Progression ────────────────────────────────────────────────
prog_rows = []
state_colors = {'Baseline': '#3B6D11', 'Partial': '#BA7517', 'Weakened': '#A32D2D'}

for state in ['Baseline', 'Partial', 'Weakened']:
    sub = fails[fails['scan_state'] == state]
    for sev in SEV_ORDER:
        c = len(sub[sub['SEVERITY'] == sev])
        prog_rows.append({'State': state, 'Severity': sev.capitalize(), 'Count': c})

prog_df = pd.DataFrame(prog_rows)

fig3 = go.Figure()
for state in ['Baseline', 'Partial', 'Weakened']:
    sub = prog_df[prog_df['State'] == state]
    total_fail = fails[fails['scan_state'] == state].shape[0]
    fig3.add_trace(go.Bar(
        name         = f'{state} (n={total_fail})',
        x            = sub['Severity'],
        y            = sub['Count'],
        marker_color = state_colors[state],
        marker_line  = dict(width=0),
        text         = sub['Count'],
        textposition = 'outside',
        textfont     = dict(size=10),
        width        = 0.24,
        hovertemplate= f'<b>{state}</b><br>%{{x}}: %{{y}}<extra></extra>'
    ))

fig3.update_layout(
    title    = dict(
        text = '<b>Vulnerability Progression Across 3 Configuration States</b><br>'
               '<sup>Baseline (140) → Partial (143) → Fully Weakened (146)</sup>',
        font = dict(size=17), x=0.02
    ),
    xaxis    = dict(title='<b>Severity Level</b>', tickfont=dict(size=13)),
    yaxis    = dict(title='<b>Number of Findings</b>', tickfont=dict(size=13),
                   range=[0, 105], gridcolor='#F0F0F0'),
    barmode  = 'group',
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    height   = 500,
    legend   = dict(title='<b>Scan State</b>', font=dict(size=12),
                    bgcolor='rgba(255,255,255,0.9)',
                    bordercolor='#CCCCCC', borderwidth=1),
    margin   = dict(t=100, b=70, l=70, r=40)
)

fig3.show()

---
## Chart 4: Top 10 Findings by Risk Index

**Research Question addressed:** RQ4 — *How can vulnerability scan results be structured into a quantitative risk prioritisation framework?*

**What this chart shows:** The 10 highest-priority findings ranked by composite Risk Index score, coloured by architectural layer.

**Key findings:**
- The S3 public access finding tops the list with Risk Index = **34.2** (Critical severity × Exposure 3 × Weight 1.2)
- The IAM cross-service confused deputy findings collectively represent the largest IAM risk cluster
- The Risk Index ranking diverges from pure CVSS ordering because it accounts for accessibility (exposure level)
- This chart directly supports the dissertation's argument that contextualised risk scoring produces superior prioritisation outcomes compared to CVSS-only ranking

**Dissertation use:** This chart populates **Table 4.4** in the Results chapter.


In [10]:
# ── Chart 4: Top 10 Risk Index Findings ──────────────────────────────────────
top10 = base.nlargest(10, 'risk_index')[
    ['CHECK_TITLE', 'risk_index', 'arch_layer', 'SEVERITY', 'cvss_score', 'exposure_level']
].copy().reset_index(drop=True)

top10['rank']  = range(1, 11)
top10['label'] = top10['CHECK_TITLE'].str[:58]
top10['color'] = top10['arch_layer'].map(COLOR_LAYER)
top10['hover'] = (
    'Rank: '          + top10['rank'].astype(str) +
    '<br>Severity: '  + top10['SEVERITY'].str.capitalize() +
    '<br>CVSS: '      + top10['cvss_score'].astype(str) +
    '<br>Exposure: '  + top10['exposure_level'].astype(str) +
    '<br>Risk Index: '+ top10['risk_index'].round(1).astype(str)
)

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    x            = top10['risk_index'][::-1],
    y            = top10['label'][::-1],
    orientation  = 'h',
    marker_color = top10['color'][::-1],
    marker_line  = dict(width=0),
    text         = top10['risk_index'][::-1].round(1).astype(str),
    textposition = 'outside',
    textfont     = dict(size=11, color='#333333'),
    hovertext    = top10['hover'][::-1],
    hoverinfo    = 'text'
))

# Add legend manually
for layer, color in COLOR_LAYER.items():
    if layer != 'Other':
        fig4.add_trace(go.Bar(
            x=[None], y=[None], orientation='h',
            name=layer, marker_color=color, showlegend=True
        ))

fig4.update_layout(
    title    = dict(
        text = '<b>Top 10 Vulnerability Findings by Risk Index Score</b><br>'
               '<sup>Risk Index = CVSS Score × Exposure Level × Architectural Weight</sup>',
        font = dict(size=17), x=0.02
    ),
    xaxis    = dict(title='<b>Risk Index Value</b>', tickfont=dict(size=12),
                   range=[0, top10['risk_index'].max() * 1.2]),
    yaxis    = dict(tickfont=dict(size=10)),
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    height   = 520,
    legend   = dict(title='<b>Layer</b>', font=dict(size=11),
                    bgcolor='rgba(255,255,255,0.9)',
                    bordercolor='#CCCCCC', borderwidth=1),
    margin   = dict(t=100, b=60, l=380, r=80),
    barmode  = 'overlay'
)
fig4.update_yaxes(gridcolor='#F0F0F0')

fig4.show()

---
## Chart 5: CVSS Score vs Exposure Level Scatter Plot

**Research Question addressed:** RQ6 — *What relationship exists between vulnerability severity (CVSS score) and exposure level?*

**What this chart shows:** A scatter plot of all 140 baseline findings plotting CVSS score (x-axis) against Exposure Level (y-axis), with a trend line and colour encoding by architectural layer.

**Key findings:**
- Pearson r = 0.545 (p < 0.001) — moderate positive correlation
- Spearman r = 0.640 (p < 0.001) — stronger rank-order correlation
- The IAM layer shows a stronger correlation (r = 0.662) than S3 (r = 0.386)
- This confirms that higher-severity vulnerabilities tend to be associated with broader exposure surfaces
- The positive relationship validates the logical foundation of the Risk Index: findings that are both severe AND broadly accessible receive the highest scores

**Statistical interpretation:** The moderate-to-strong correlation justifies combining CVSS and Exposure Level in the Risk Index formula.


In [11]:
# ── Chart 5: CVSS vs Exposure Scatter ────────────────────────────────────────
fig5 = go.Figure()

# Plot points by layer
for layer in ['IAM Access Control', 'S3 Storage']:
    sub = base[base['arch_layer'] == layer]
    fig5.add_trace(go.Scatter(
        x    = sub['cvss_score'] + np.random.uniform(-0.08, 0.08, len(sub)),  # jitter
        y    = sub['exposure_level'] + np.random.uniform(-0.06, 0.06, len(sub)),
        mode = 'markers',
        name = layer,
        marker = dict(color=COLOR_LAYER[layer], size=9, opacity=0.65,
                      line=dict(width=0.5, color='white')),
        hovertemplate = (
            f'<b>{layer}</b><br>'
            'CVSS: %{x:.1f}<br>'
            'Exposure: %{y:.0f}<extra></extra>'
        )
    ))

# Trend line (OLS)
clean5 = base[['cvss_score', 'exposure_level']].dropna()
m5, b5 = np.polyfit(clean5['cvss_score'], clean5['exposure_level'], 1)
xs5    = np.linspace(clean5['cvss_score'].min(), clean5['cvss_score'].max(), 100)
r_val, _ = pearsonr(clean5['cvss_score'], clean5['exposure_level'])

fig5.add_trace(go.Scatter(
    x    = xs5, y = m5 * xs5 + b5,
    mode = 'lines',
    name = f'Trend line (r = {r_val:.3f})',
    line = dict(color='#444444', width=2, dash='dash')
))

fig5.update_layout(
    title    = dict(
        text = '<b>CVSS Score vs Exposure Level</b><br>'
               f'<sup>Pearson r = {r_val:.3f} | p < 0.001 | n = {len(clean5)} findings</sup>',
        font = dict(size=17), x=0.02
    ),
    xaxis    = dict(
        title      = '<b>CVSS v3 Score</b>',
        tickfont   = dict(size=13),
        tickvals   = [2.5, 5.5, 7.5, 9.5],
        ticktext   = ['2.5 (Low)', '5.5 (Medium)', '7.5 (High)', '9.5 (Critical)'],
        gridcolor  = '#F0F0F0'
    ),
    yaxis    = dict(
        title    = '<b>Exposure Level</b>',
        tickfont = dict(size=13),
        tickvals = [1, 2, 3],
        ticktext = ['1 — Internal', '2 — Cross-account', '3 — Public'],
        gridcolor= '#F0F0F0',
        range    = [0.5, 3.5]
    ),
    plot_bgcolor  = 'white',
    paper_bgcolor = 'white',
    height   = 500,
    legend   = dict(font=dict(size=12), bgcolor='rgba(255,255,255,0.9)',
                    bordercolor='#CCCCCC', borderwidth=1),
    margin   = dict(t=100, b=80, l=140, r=40)
)

fig5.show()

---
## Inferential Statistical Tests

This cell runs all four statistical tests referenced in the dissertation methodology (Section 3.5).

### Tests Performed

| Test | Purpose | Expected Result |
|---|---|---|
| **Chi-square** | Is severity distribution significantly different between IAM and S3 layers? | p < 0.05 = significant difference |
| **Pearson r** | Linear correlation between CVSS score and Exposure Level | r > 0.5 = moderate relationship |
| **Spearman r** | Rank-order correlation (robust to non-normality) | Should be higher than Pearson for ordinal data |
| **Linear Regression** | Can we predict Risk Index from CVSS, Exposure, and Weight? | R² > 0.9 = strong model fit |

### Interpreting Results for the Dissertation
- **Cramer's V** (effect size for chi-square): 0.1=small, 0.3=medium, 0.5=large
- **Correlation strength**: 0.1=weak, 0.3=moderate, 0.5=strong, 0.7=very strong
- **R-squared**: proportion of variance in Risk Index explained by the three predictors


In [12]:
print('=' * 65)
print('INFERENTIAL STATISTICAL TESTS — DISSERTATION RESULTS')
print('=' * 65)

# ── 1. Chi-Square Test ────────────────────────────────────────────────────────
print('\n1. CHI-SQUARE TEST: Severity Distribution vs Architectural Layer')
print('   H0: Severity distribution is independent of architectural layer')
print('   H1: Severity distribution differs significantly between layers')
ct = pd.crosstab(base['SEVERITY'], base['arch_layer'])
print('\n   Contingency Table:')
print('   ' + ct.to_string().replace('\n', '\n   '))
chi2, p, dof, expected = chi2_contingency(ct)
n        = ct.values.sum()
cramersv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f'\n   chi2       = {chi2:.3f}')
print(f'   df         = {dof}')
print(f'   p-value    = {p:.6f}  {"*** SIGNIFICANT" if p < 0.001 else "** SIGNIFICANT" if p < 0.01 else "* SIGNIFICANT" if p < 0.05 else "NOT significant"}')
print(f"\n   Cramer's V = {cramersv:.3f}  ({'Large effect' if cramersv > 0.5 else 'Medium effect' if cramersv > 0.3 else 'Small effect'})\n")
print(f'   Conclusion : {"Reject H0" if p < 0.05 else "Fail to reject H0"} — severity distribution differs significantly between layers')

# ── 2. Pearson Correlation ────────────────────────────────────────────────────
print('\n2. PEARSON CORRELATION: CVSS Score vs Exposure Level')
clean2 = base[['cvss_score', 'exposure_level']].dropna()
r_p, p_p = pearsonr(clean2['cvss_score'], clean2['exposure_level'])
r_s, p_s = spearmanr(clean2['cvss_score'], clean2['exposure_level'])
print(f'   Pearson  r = {r_p:.3f}, p = {p_p:.6f}  {"*** SIGNIFICANT" if p_p < 0.001 else ""}')
print(f'   Spearman r = {r_s:.3f}, p = {p_s:.6f}  {"*** SIGNIFICANT" if p_s < 0.001 else ""}')
print(f'   Strength   : {"Strong" if abs(r_p) > 0.5 else "Moderate" if abs(r_p) > 0.3 else "Weak"} positive correlation')

print('\n   Stratified by Layer:')
for layer in ['IAM Access Control', 'S3 Storage']:
    sub = base[base['arch_layer'] == layer][['cvss_score', 'exposure_level']].dropna()
    if len(sub) > 5:
        r, p2 = pearsonr(sub['cvss_score'], sub['exposure_level'])
        print(f'   [{layer}] r = {r:.3f}, p = {p2:.4f}, n = {len(sub)}')

# ── 3. Linear Regression ──────────────────────────────────────────────────────
print('\n3. LINEAR REGRESSION: Predicting Risk Index')
print('   Predictors: CVSS Score, Exposure Level, Architectural Weight')
print('   Target    : Risk Index')
reg_data = base[['cvss_score', 'exposure_level', 'arch_weight', 'risk_index']].dropna()
X   = reg_data[['cvss_score', 'exposure_level', 'arch_weight']]
y   = reg_data['risk_index']
reg = LinearRegression().fit(X, y)
y_pred = reg.predict(X)
r2   = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f'\n   R-squared     = {r2:.3f}  ({r2*100:.1f}% of variance explained)')
print(f'   RMSE          = {rmse:.3f}')
print(f'   Intercept     = {reg.intercept_:.3f}')
print(f'   Coeff (CVSS)  = {reg.coef_[0]:.3f}')
print(f'   Coeff (Exp.)  = {reg.coef_[1]:.3f}')
print(f'   Coeff (Weight)= {reg.coef_[2]:.3f}')

print('\n' + '=' * 65)
print('ALL STATISTICAL TESTS COMPLETE')
print('=' * 65)

INFERENTIAL STATISTICAL TESTS — DISSERTATION RESULTS

1. CHI-SQUARE TEST: Severity Distribution vs Architectural Layer
   H0: Severity distribution is independent of architectural layer
   H1: Severity distribution differs significantly between layers

   Contingency Table:
   arch_layer  IAM Access Control  S3 Storage
   SEVERITY                                  
   critical                     0           1
   high                        20           1
   low                         16          13
   medium                      26          63

   chi2       = 32.478
   df         = 3
   p-value    = 0.000000  *** SIGNIFICANT

   Cramer's V = 0.482  (Medium effect)

   Conclusion : Reject H0 — severity distribution differs significantly between layers

2. PEARSON CORRELATION: CVSS Score vs Exposure Level
   Pearson  r = 0.545, p = 0.000000  *** SIGNIFICANT
   Spearman r = 0.640, p = 0.000000  *** SIGNIFICANT
   Strength   : Strong positive correlation

   Stratified by Layer:
   [IAM 

---
## Risk Index Prioritisation Table

This cell generates the **Top 10 Risk Index findings table** that maps directly to **Table 4.4** in the dissertation.

The Risk Index ranking is compared against a pure CVSS-ordered ranking to demonstrate the value of contextualised prioritisation. Findings that rank differently between the two approaches highlight cases where exposure level changes the priority assessment.

**How to use this table in the dissertation:**
- Copy the printed table into Table 4.4
- Highlight any rows where Risk Index rank differs from CVSS rank
- Use these differences to argue for Risk Index superiority in Section 4.5


In [13]:
# ── Top 10 Risk Index Table ───────────────────────────────────────────────────
top10_tbl = base.nlargest(10, 'risk_index')[
    ['CHECK_TITLE', 'SEVERITY', 'cvss_score', 'exposure_level', 'arch_weight', 'risk_index', 'arch_layer']
].copy().reset_index(drop=True)

top10_tbl.index = range(1, 11)
top10_tbl.index.name = 'Rank'

# Truncate long titles
top10_tbl['CHECK_TITLE'] = top10_tbl['CHECK_TITLE'].str[:60]

print('TABLE 4.4 — Top 10 Findings by Risk Index (Dissertation)')
print('=' * 95)
print(f'{"Rank":<5} {"Finding":<62} {"Sev":<10} {"CVSS":<6} {"Exp":<5} {"Wt":<5} {"RI":<7} {"Layer":<22}')
print('-' * 95)
for i, row in top10_tbl.iterrows():
    print(f'{i:<5} {row["CHECK_TITLE"]:<62} {row["SEVERITY"].capitalize():<10} '
          f'{row["cvss_score"]:<6.1f} {row["exposure_level"]:<5} {row["arch_weight"]:<5.1f} '
          f'{row["risk_index"]:<7.1f} {row["arch_layer"]:<22}')
print('=' * 95)
print(f'\nNote: RI = Risk Index = CVSS × Exposure × Weight')
print(f'Maximum possible Risk Index = 9.5 × 3 × 1.5 = 42.75')

TABLE 4.4 — Top 10 Findings by Risk Index (Dissertation)
Rank  Finding                                                        Sev        CVSS   Exp   Wt    RI      Layer                 
-----------------------------------------------------------------------------------------------
1     Ensure there are no S3 buckets open to Everyone or Any AWS u   Critical   9.5    3     1.2   34.2    S3 Storage            
2     Check S3 Account Level Public Access Block.                    High       7.5    3     1.2   27.0    S3 Storage            
3     Ensure IAM Service Roles prevents against a cross-service co   High       7.5    2     1.5   22.5    IAM Access Control    
4     Ensure IAM Service Roles prevents against a cross-service co   High       7.5    2     1.5   22.5    IAM Access Control    
5     Ensure IAM Service Roles prevents against a cross-service co   High       7.5    2     1.5   22.5    IAM Access Control    
6     Ensure IAM Service Roles prevents against a cross-service co 

---
## Business Impact Analysis

This cell translates technical vulnerability findings into financial business metrics for inclusion in **Section 4.6** of the dissertation.

### Methodology
1. **Breach probability mapping** — Critical findings are assigned a 35% annual breach probability, High=15%, Medium=5%, Low=1% (based on Ponemon Institute benchmarks for cloud environments)
2. **Breach cost** — Mean breach cost of USD 4.88 million for cloud-hosted environments (IBM Cost of a Data Breach Report 2024)
3. **Remediation cost** — Industry benchmark effort estimates per finding category
4. **ROI calculation** — Risk-adjusted return on remediation investment

### Important Note
These figures are estimates based on published industry benchmarks. They should be presented in the dissertation as illustrative financial framing rather than precise predictions.


In [14]:
print('=' * 65)
print('BUSINESS IMPACT ANALYSIS')
print('=' * 65)

# ── Breach probability and cost parameters ────────────────────────────────────
breach_prob   = {'critical': 0.35, 'high': 0.15, 'medium': 0.05, 'low': 0.01}
mean_breach_cost_usd = 4_880_000  # IBM 2024

# ── Expected loss per finding ─────────────────────────────────────────────────
base_calc = base.copy()
base_calc['breach_prob'] = base_calc['SEVERITY'].map(breach_prob)
base_calc['expected_loss'] = base_calc['breach_prob'] * mean_breach_cost_usd

# ── Summary by severity ───────────────────────────────────────────────────────
print('\n1. EXPECTED BREACH COST BY SEVERITY (Baseline)')
print(f'   Mean breach cost (IBM 2024): USD {mean_breach_cost_usd:,.0f}')
print(f'\n   {"Severity":<12} {"Count":>6} {"Breach Prob":>12} {"Expected Loss (USD)":>22}')
print('   ' + '-' * 55)
total_exposure = 0
for sev in SEV_ORDER:
    sub = base_calc[base_calc['SEVERITY'] == sev]
    if len(sub) > 0:
        prob = breach_prob[sev]
        exp_loss = len(sub) * prob * mean_breach_cost_usd
        total_exposure += exp_loss
        print(f'   {sev.capitalize():<12} {len(sub):>6} {prob:>11.0%} {exp_loss:>21,.0f}')
print('   ' + '-' * 55)
print(f'   {"TOTAL":<12} {len(base_calc):>6} {"":>12} {total_exposure:>21,.0f}')

# ── Critical + High combined ──────────────────────────────────────────────────
crit_high = base_calc[base_calc['SEVERITY'].isin(['critical', 'high'])]
crit_high_loss = (crit_high['breach_prob'] * mean_breach_cost_usd).sum()
print(f'\n   Critical + High combined exposure: USD {crit_high_loss:,.0f}')

# ── Top 10 remediation ROI ────────────────────────────────────────────────────
print('\n2. REMEDIATION ROI — TOP 10 RISK INDEX FINDINGS')
remediation_cost_per_finding = {
    'critical': 8000, 'high': 4500, 'medium': 1500, 'low': 500
}
top10_roi = base_calc.nlargest(10, 'risk_index').copy()
top10_roi['rem_cost']    = top10_roi['SEVERITY'].map(remediation_cost_per_finding)
top10_roi['risk_reduce'] = top10_roi['breach_prob'] * mean_breach_cost_usd
total_rem_cost    = top10_roi['rem_cost'].sum()
total_risk_reduce = top10_roi['risk_reduce'].sum()
roi_ratio         = total_risk_reduce / total_rem_cost

print(f'\n   Total remediation cost (Top 10) : USD {total_rem_cost:>10,.0f}')
print(f'   Total risk exposure reduced     : USD {total_risk_reduce:>10,.0f}')
print(f'   ROI ratio                       : {roi_ratio:.1f}x')
print(f'   For every USD 1 spent on remediating the top 10 findings,')
print(f'   USD {roi_ratio:.0f} of expected breach cost is avoided.')

# ── Risk reduction percentage ─────────────────────────────────────────────────
total_ri_all  = base_calc['risk_index'].sum()
total_ri_top10= top10_roi['risk_index'].sum()
pct_reduction = total_ri_top10 / total_ri_all * 100
print(f'\n3. RISK INDEX REDUCTION')
print(f'   Total Risk Index (all 140 findings) : {total_ri_all:.1f}')
print(f'   Risk Index of Top 10 findings       : {total_ri_top10:.1f}')
print(f'   Remediating Top 10 reduces aggregate')
print(f'   Risk Index by                       : {pct_reduction:.1f}%')

print('\n' + '=' * 65)
print('BUSINESS IMPACT ANALYSIS COMPLETE')
print('=' * 65)

BUSINESS IMPACT ANALYSIS

1. EXPECTED BREACH COST BY SEVERITY (Baseline)
   Mean breach cost (IBM 2024): USD 4,880,000

   Severity      Count  Breach Prob    Expected Loss (USD)
   -------------------------------------------------------
   Critical          1         35%             1,708,000
   High             21         15%            15,372,000
   Medium           89          5%            21,716,000
   Low              29          1%             1,415,200
   -------------------------------------------------------
   TOTAL           140                         40,211,200

   Critical + High combined exposure: USD 17,080,000

2. REMEDIATION ROI — TOP 10 RISK INDEX FINDINGS

   Total remediation cost (Top 10) : USD     48,500
   Total risk exposure reduced     : USD  8,296,000
   ROI ratio                       : 171.1x
   For every USD 1 spent on remediating the top 10 findings,
   USD 171 of expected breach cost is avoided.

3. RISK INDEX REDUCTION
   Total Risk Index (all 140 fin

---
## Cell 14 — Summary of Key Findings

This final cell prints a consolidated summary of all key findings for easy copy-paste into the dissertation Results and Conclusion chapters.


In [15]:
print('=' * 65)
print('CONSOLIDATED FINDINGS SUMMARY FOR DISSERTATION')
print('=' * 65)

print('''
CHAPTER 4 — KEY FINDINGS SUMMARY

4.1 Vulnerability Distribution (Scan 1 Baseline, n=140):
    • Critical : 1   (0.7%)
    • High     : 21  (15.0%)
    • Medium   : 89  (63.6%)
    • Low      : 29  (20.7%)
    • Mean CVSS: 5.21 (SD=1.58)

4.2 Architectural Layer Analysis:
    • IAM layer: 62 failures → 0 Critical, 20 High, 26 Medium, 16 Low
    • S3 layer : 78 failures → 1 Critical,  1 High, 63 Medium, 13 Low
    • IAM dominates High severity (20 vs 1)
    • Answer to RQ3: IAM layer has highest concentration of high-severity findings

4.3 Scan Progression:
    • Baseline → Partial → Weakened: 140 → 143 → 146 failures
    • S3 Medium findings increased by +3 per weakening step
    • Demonstrates Prowler's sensitivity to configuration changes

4.4 Statistical Tests:
    • Chi-square: χ²=32.478, df=3, p<0.001, Cramer's V=0.482 (medium-large effect)
    • Pearson  r: 0.545 (p<0.001) — CVSS vs Exposure Level
    • Spearman r: 0.640 (p<0.001)
    • R-squared : 0.984 — Risk Index regression model fit

4.5 Top Risk Index Finding:
    • S3 public access → Risk Index = 34.2 (Critical severity)
    • IAM cross-service confused deputy → Risk Index = 22.5 (High severity)

4.6 Business Impact:
    • Total aggregate breach cost exposure: see Cell 13 output
    • Remediating Top 10 findings reduces aggregate Risk Index by ~35%
    • ROI: see Cell 13 output
''')

print('\nNotebook analysis complete!')

CONSOLIDATED FINDINGS SUMMARY FOR DISSERTATION

CHAPTER 4 — KEY FINDINGS SUMMARY

4.1 Vulnerability Distribution (Scan 1 Baseline, n=140):
    • Critical : 1   (0.7%)
    • High     : 21  (15.0%)
    • Medium   : 89  (63.6%)
    • Low      : 29  (20.7%)
    • Mean CVSS: 5.21 (SD=1.58)

4.2 Architectural Layer Analysis:
    • IAM layer: 62 failures → 0 Critical, 20 High, 26 Medium, 16 Low
    • S3 layer : 78 failures → 1 Critical,  1 High, 63 Medium, 13 Low
    • IAM dominates High severity (20 vs 1)
    • Answer to RQ3: IAM layer has highest concentration of high-severity findings

4.3 Scan Progression:
    • Baseline → Partial → Weakened: 140 → 143 → 146 failures
    • S3 Medium findings increased by +3 per weakening step
    • Demonstrates Prowler's sensitivity to configuration changes

4.4 Statistical Tests:
    • Chi-square: χ²=32.478, df=3, p<0.001, Cramer's V=0.482 (medium-large effect)
    • Pearson  r: 0.545 (p<0.001) — CVSS vs Exposure Level
    • Spearman r: 0.640 (p<0.001)
 